# TP 3 - Agent PydanticAI


---
## 0. Configuration partagée


In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel, GoogleModelSettings
from pydantic_ai.providers.google import GoogleProvider

from shared.agent_utils import run_agent_realtime_logging
from shared.agent_tools import tool_get_current_date, tool_geocode_location, tool_get_weather, tool_search_nearby, tool_retrieve_docs, web_search, web_extract
from shared.config import ROOT_DIR, project_settings

**TODO — Configuration partagée et périmètre de code**

Fichiers à modifier :
- `shared/agent_tools.py`
- `TP3_travel_planner_Agent/3_1_tooling_assistant.ipynb`

La fonction principale à coder est `run_agent_realtime_logging` qui exécute un agent avec un prompt donné, en enregistrant la trace complète (étapes, outils appelés, réponses). La partie traceback est déjà fournie dans `shared/agent_utils.py`.

Ce TP s'appuie aussi sur plusieurs outils externes et internes :

`tool_get_current_date` : fonction qui convertit les dates relatives en dates calendrier explicites

`tool_geocode_location` : fonction qui transforme un lieu texte en coordonnées géographiques

`tool_get_weather` : fonction qui retourne une prévision météo sur une plage de dates

`tool_retrieve_docs` : fonction qui récupère les passages RAG internes les plus pertinents

`tool_search_nearby` : fonction qui cherche des lieux proches autour de coordonnées avec filtres

`web_search` : fonction qui lance une recherche web externe

`web_extract` : fonction qui lit le contenu d'une page web trouvée

`Place` : classe qui représente un lieu avec un nom et des coordonnées

Vous pouvez coder ces fonctions en même temps que le développement de l'agent, au fur et à mesure des cas d'usage.


In [ ]:
# TODO : configurer le modèle et les settings partagés pour tous les use cases
model = GoogleModel(
    model_name=project_settings.llm_model_name,
    provider=GoogleProvider(api_key=project_settings.google_api_key),
)

agent_model_settings = GoogleModelSettings(
    temperature=project_settings.llm_temperature,
    top_p=project_settings.llm_top_p,
    max_tokens=project_settings.llm_max_output_tokens,
    google_thinking_config={"thinking_budget": project_settings.llm_thinking_budget},
)


Compléter le prompt système ci-dessous, nous y ajouterons en dessous les différentes consignes d'utilisation des outils au fur et à mesure des cas d'usage.

Conseil : Demandez à l'agent de sourcer chaque affirmation factuelle avec le résultat d'outil, par exemple en ajoutant `[tool_name : source]` à la fin de chaque fait. (Exemple : `[web_search : wikipedia.com/Rome]`)

In [ ]:
# TODO : Écrire le prompt système de base, qui sera complété par les guides d'utilisation des différents outils au fur et à mesure du TP

base_system_prompt = """

# RÈGLES

Tu es un assistant de planification de voyage factuel.
Tu dois produire des recommandations personnalisées à partir de faits récupérés par outils.

## Règles générales

### Usage des outils et du contexte
- Utiliser les outils avant de répondre dès qu'un fait est nécessaire.
- Chaîner les outils si besoin (date -> geocode -> météo, docs -> web_search -> web_extract, etc.).
- N'utiliser que des faits issus des outils et du contexte récupéré.
- Citer les preuves factuelles (ex: [tool_retrieve_docs:source], [web_search:url]).
- Si les données sont partielles ou contradictoires, le dire explicitement.
- Si une information manque, la lister clairement sans inventer.
- Ignorer le contexte hors sujet.

### Style de réponse
- Réponse concise et précise.
- Pas de Markdown décoratif.
- Pour chaque recommandation: 1 raison + 1 détail pratique (horaire, lieu, budget, logistique).
- Préférer des actions concrètes aux formulations vagues.

### Format attendu
1) Résumé: réponse directe
2) Plan: séquence concrète (jour par jour ou par objectif)
3) Budget: estimation par catégorie uniquement sur faits disponibles
4) Preuves: faits clés utilisés (sources/outils)
5) Informations manquantes: liste courte et explicite

Important: ne pas interrompre avec des questions en cours de route. Produire la meilleure réponse possible avec les données disponibles, puis lister les manques.

## Consignes spécifiques par outil

"""


---

## 1. Cas d'usage 1 - Rome en 4 jours

**Objectif** : Prompter et outiller l'agent pour résoudre l'use case que nous avons abordé sur les autres TP.

Exemple d'ordre d'utilisation des outils : date -> géocodage -> météo -> docs -> réponse.
(**Ne pas hard-coder l'ordre ou l'utilisatio** spécifique d'outils, l'agent doit pouvoir définir par lui-même la stratégie de recherche et d'appel d'outils !)

Vérification concrète dans la trace: chaque recommandation doit être justifiée par au moins un résultat d'outil et respecter durée/budget/préférences.

In [ ]:
prompt_use_case_1 = (
    "Je vais à Rome la semaine prochaine pour 4 jours (du jeudi au dimanche), arrivée le matin, départ le soir. "
    "Fais un plan de 4 jours avec un budget de 300 EUR pour les sorties et restaurants. "
    "Évite les zones trop touristiques et privilégie les lieux confidentiels."
)

In [ ]:
system_prompt_extended = base_system_prompt + """
### tool_get_current_date
- Utiliser cet outil en premier si la demande contient des dates relatives (semaine prochaine, ce week-end, etc.)
- Convertir les dates relatives en dates calendrier explicites avant tout appel dépendant des dates

### tool_geocode_location
- Utiliser cet outil quand l'utilisateur fournit un lieu texte (ville, adresse, quartier)
- Réutiliser les coordonnées retournées pour météo et nearby

### tool_get_weather
- Utiliser cet outil avec coordonnées + dates ISO précises
- Expliquer l'impact météo sur l'ordre des activités et la faisabilité budget

### tool_retrieve_docs
- Utiliser des requêtes ciblées pour récupérer des recommandations locales
- Si la première récupération est faible, reformuler puis relancer
- Ne garder que les preuves réellement utiles à la réponse finale

"""

tools_use_case_1 = [tool_get_current_date, tool_geocode_location, tool_get_weather, tool_retrieve_docs]

agent_use_case_1 = Agent(
    model=model,
    instructions=system_prompt_extended,
    model_settings=agent_model_settings,
    tools=tools_use_case_1,
)

result_use_case_1 = await run_agent_realtime_logging(
    agent=agent_use_case_1,
    prompt=prompt_use_case_1,
    log_path=ROOT_DIR / "TP3_travel_planner_Agent" / "logs" / "trace_use_case_1.log",
    max_steps=12,
)


In [ ]:
print(result_use_case_1.output[:1200])


---

## 2. Cas d'usage 2 - Meilleure période Paris -> New York

Objectif : On montre la polyvalence de l'agent en attaquant d'autres types de questions dans le même domaine de la planification de voyage.

On peut tester plusieurs questions tant que l'agent est capable d'y répondre avec les outils qu'on lui donne !

Dans cet exemple, l'agent doit faire un raisonnement long horizon (sur plusieurs mois).
`tool_get_weather` ne sert que pour le court terme : la preuve principale doit venir de `web_search` (pour chercher des pages web) et `web_extract` (pour lire des pages web).

Vous pouvez insister sur le fait de faire plusieurs recherches web afin de garantir des meilleurs résultats.

In [ ]:
prompt_use_case_2 = (
    "Trouve la meilleure période dans les 6 prochains mois pour un voyage Paris -> New York. "
    "Compare météo et informations web sur les prix saisonniers, et justifie la recommandation."
    "Précise les prix, le temps de trajet et les conditions météo."
)

In [ ]:
system_prompt_extended = base_system_prompt + """

### tool_get_weather
- Outil réservé aux prévisions court terme
- Ne pas l'utiliser pour prédire la météo à 6 mois ou un climat de saison
- Pour un choix de période long terme, utiliser `web_search` et `web_extract` comme preuves principales
- Utiliser la météo seulement comme signal complémentaire si les dates sont dans la fenêtre supportée

### web_search
- Utiliser cet outil pour collecter des sources publiques récentes sur période, coûts et tendances
- Construire des requêtes ciblées (villes, mois, saison, budget)

### web_extract
- Utiliser cet outil sur les URL les plus prometteuses trouvées via `web_search`
- Extraire uniquement les pages utiles, pas tous les résultats

"""

tools_use_case_2 = tools_use_case_1 + [web_search, web_extract]

agent_use_case_2 = Agent(
    model=model,
    instructions=system_prompt_extended,
    model_settings=agent_model_settings,
    tools=tools_use_case_2,
)

result_use_case_2 = await run_agent_realtime_logging(
    agent=agent_use_case_2,
    prompt=prompt_use_case_2,
    log_path=ROOT_DIR / "TP3_travel_planner_Agent" / "logs" / "trace_use_case_2.log",
    max_steps=12,
)

In [ ]:
print(result_use_case_2.output[:1200])

---

## 3. Cas d'usage 3 - Recommandations proches depuis une adresse


In [ ]:
prompt_use_case_3 = (
    "Recommande les meilleurs restaurants et activités près de cette adresse : "
    "10 Rue de la Paix, 75002 Paris, France. " # Vous pouvz tester avec d'autres adresses
    "J'ai un budget de 30 euros pour un repas et 20 euros pour une activité. "
)

In [ ]:
system_prompt_extended = base_system_prompt + """

### tool_geocode_location
- Utiliser cet outil en premier pour convertir l'adresse texte en coordonnées
- Vérifier la cohérence du lieu avant de lancer nearby

### tool_search_nearby
- Choisir un `place_type` aligné avec l'intention (restaurant, museum, park, etc.)
- Ajouter un `keyword` quand un filtrage fin est nécessaire (calme, local, rooftop, etc.)
- Garder une sélection finale strictement alignée avec les contraintes utilisateur

"""

tools_use_case_3 = tools_use_case_2 + [tool_search_nearby]

agent_use_case_3 = Agent(
    model=model,
    instructions=system_prompt_extended,
    model_settings=agent_model_settings,
    tools=tools_use_case_3,
)

result_use_case_3 = await run_agent_realtime_logging(
    agent=agent_use_case_3,
    prompt=prompt_use_case_3,
    log_path=ROOT_DIR / "TP3_travel_planner_Agent" / "logs" / "trace_use_case_3.log",
    max_steps=12,
)

In [ ]:
print(result_use_case_3.output[:1200])